In [7]:
from pathlib import Path
import os
import sys
import shutil
import random
from PIL import Image
from ultralytics import YOLO

PROJECT_ROOT = Path(r"C:\Users\User\Desktop\Vehicle_Damage_Detection")
os.chdir(PROJECT_ROOT)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

print("Current folder:", Path.cwd())
print("Project exists:", PROJECT_ROOT.exists())

Current folder: C:\Users\User\Desktop\Vehicle_Damage_Detection
Project exists: True


In [8]:
DAMAGE_MODEL_PATH = PROJECT_ROOT / "models" / "detection" / "cardd_coco_best.pt"
CARPARTS_MODEL_PATH = PROJECT_ROOT / "models" / "segmentation" / "carparts_best.pt"

OUTPUT_DIR = PROJECT_ROOT / "data" / "processed" / "damaged_parts_yolo"

SOURCE_IMAGE_DIRS = [
    PROJECT_ROOT / "data" / "processed" / "cardd_coco_yolo" / "images" / "train",
    PROJECT_ROOT / "data" / "processed" / "cardd_coco_yolo" / "images" / "val",

    PROJECT_ROOT / "data" / "processed" / "cardd_sod_yolo" / "images" / "train",
    PROJECT_ROOT / "data" / "processed" / "cardd_sod_yolo" / "images" / "val",

    PROJECT_ROOT / "data" / "processed" / "archive4_yolo" / "images" / "train",
    PROJECT_ROOT / "data" / "processed" / "archive4_yolo" / "images" / "val",
]

DAMAGED_PART_CLASSES = [
    "front_bumper_damage",
    "rear_bumper_damage",
    "hood_damage",
    "front_light_damage",
    "rear_light_damage",
    "front_door_damage",
    "rear_door_damage",
    "side_mirror_damage",
    "windshield_damage",
    "wheel_tire_damage",
]

CLASS_TO_ID = {name: idx for idx, name in enumerate(DAMAGED_PART_CLASSES)}

PART_TO_DAMAGED_CLASS = {
    "front_bumper": "front_bumper_damage",
    "back_bumper": "rear_bumper_damage",

    "hood": "hood_damage",

    "front_light": "front_light_damage",
    "front_left_light": "front_light_damage",
    "front_right_light": "front_light_damage",

    "back_light": "rear_light_damage",
    "back_left_light": "rear_light_damage",
    "back_right_light": "rear_light_damage",

    "front_door": "front_door_damage",
    "front_left_door": "front_door_damage",
    "front_right_door": "front_door_damage",

    "back_door": "rear_door_damage",
    "back_left_door": "rear_door_damage",
    "back_right_door": "rear_door_damage",

    "left_mirror": "side_mirror_damage",
    "right_mirror": "side_mirror_damage",

    "front_glass": "windshield_damage",
    "back_glass": "windshield_damage",

    "wheel": "wheel_tire_damage",
}

print("Damage model exists:", DAMAGE_MODEL_PATH.exists())
print("Carparts model exists:", CARPARTS_MODEL_PATH.exists())

for path in SOURCE_IMAGE_DIRS:
    print(path, "exists:", path.exists())

Damage model exists: True
Carparts model exists: True
C:\Users\User\Desktop\Vehicle_Damage_Detection\data\processed\cardd_coco_yolo\images\train exists: True
C:\Users\User\Desktop\Vehicle_Damage_Detection\data\processed\cardd_coco_yolo\images\val exists: True
C:\Users\User\Desktop\Vehicle_Damage_Detection\data\processed\cardd_sod_yolo\images\train exists: True
C:\Users\User\Desktop\Vehicle_Damage_Detection\data\processed\cardd_sod_yolo\images\val exists: True
C:\Users\User\Desktop\Vehicle_Damage_Detection\data\processed\archive4_yolo\images\train exists: True
C:\Users\User\Desktop\Vehicle_Damage_Detection\data\processed\archive4_yolo\images\val exists: True


In [9]:
def calculate_iou(box_a, box_b):
    x1_a, y1_a, x2_a, y2_a = box_a
    x1_b, y1_b, x2_b, y2_b = box_b

    inter_x1 = max(x1_a, x1_b)
    inter_y1 = max(y1_a, y1_b)
    inter_x2 = min(x2_a, x2_b)
    inter_y2 = min(y2_a, y2_b)

    inter_w = max(0, inter_x2 - inter_x1)
    inter_h = max(0, inter_y2 - inter_y1)
    intersection = inter_w * inter_h

    area_a = max(0, x2_a - x1_a) * max(0, y2_a - y1_a)
    area_b = max(0, x2_b - x1_b) * max(0, y2_b - y1_b)

    union = area_a + area_b - intersection

    if union == 0:
        return 0.0

    return intersection / union


def box_center(box):
    x1, y1, x2, y2 = box
    return [(x1 + x2) / 2, (y1 + y2) / 2]


def center_inside_box(center, box):
    cx, cy = center
    x1, y1, x2, y2 = box
    return x1 <= cx <= x2 and y1 <= cy <= y2


def yolo_box_from_xyxy(box, image_width, image_height):
    x1, y1, x2, y2 = box

    x_center = ((x1 + x2) / 2) / image_width
    y_center = ((y1 + y2) / 2) / image_height
    width = (x2 - x1) / image_width
    height = (y2 - y1) / image_height

    return x_center, y_center, width, height


def extract_boxes(result, names):
    items = []

    if result.boxes is None:
        return items

    for box in result.boxes:
        class_id = int(box.cls[0])
        confidence = float(box.conf[0])
        xyxy = box.xyxy[0].tolist()

        items.append({
            "class_name": names[class_id],
            "confidence": confidence,
            "box": xyxy
        })

    return items

In [10]:
damage_model = YOLO(str(DAMAGE_MODEL_PATH))
carparts_model = YOLO(str(CARPARTS_MODEL_PATH))

print("Damage model classes:")
print(damage_model.names)

print("\nCarparts model classes:")
print(carparts_model.names)

Damage model classes:
{0: 'dent', 1: 'scratch', 2: 'crack', 3: 'glass shatter', 4: 'lamp broken', 5: 'tire flat'}

Carparts model classes:
{0: 'back_bumper', 1: 'back_door', 2: 'back_glass', 3: 'back_left_door', 4: 'back_left_light', 5: 'back_light', 6: 'back_right_door', 7: 'back_right_light', 8: 'front_bumper', 9: 'front_door', 10: 'front_glass', 11: 'front_left_door', 12: 'front_left_light', 13: 'front_light', 14: 'front_right_door', 15: 'front_right_light', 16: 'hood', 17: 'left_mirror', 18: 'object', 19: 'right_mirror', 20: 'tailgate', 21: 'trunk', 22: 'wheel'}


In [12]:
image_paths = []

for folder in SOURCE_IMAGE_DIRS:
    if folder.exists():
        image_paths.extend(list(folder.glob("*.jpg")))
        image_paths.extend(list(folder.glob("*.jpeg")))
        image_paths.extend(list(folder.glob("*.png")))

random.shuffle(image_paths)

# Start with max 500 images to save time
image_paths = image_paths[:2000]

print("Images selected for auto-labelling:", len(image_paths))
print("Sample:", image_paths[:3])

Images selected for auto-labelling: 2000
Sample: [WindowsPath('C:/Users/User/Desktop/Vehicle_Damage_Detection/data/processed/cardd_sod_yolo/images/train/003061.jpg'), WindowsPath('C:/Users/User/Desktop/Vehicle_Damage_Detection/data/processed/archive4_yolo/images/train/03012020_111113image823703.jpg'), WindowsPath('C:/Users/User/Desktop/Vehicle_Damage_Detection/data/processed/cardd_sod_yolo/images/train/002927.jpg')]


In [13]:
# Clear old generated dataset if exists
if OUTPUT_DIR.exists():
    shutil.rmtree(OUTPUT_DIR)

for split in ["train", "val"]:
    (OUTPUT_DIR / "images" / split).mkdir(parents=True, exist_ok=True)
    (OUTPUT_DIR / "labels" / split).mkdir(parents=True, exist_ok=True)

print("Output folder ready:", OUTPUT_DIR)

Output folder ready: C:\Users\User\Desktop\Vehicle_Damage_Detection\data\processed\damaged_parts_yolo


In [14]:
MIN_DAMAGE_CONF = 0.30
MIN_PART_CONF = 0.30
MIN_IOU = 0.05

created_images = 0
created_labels = 0
skipped_no_label = 0

for idx, image_path in enumerate(image_paths):
    split = "train" if random.random() < 0.8 else "val"

    try:
        image = Image.open(image_path).convert("RGB")
        image_width, image_height = image.size
    except Exception:
        continue

    damage_result = damage_model.predict(
        source=str(image_path),
        conf=MIN_DAMAGE_CONF,
        verbose=False
    )[0]

    carparts_result = carparts_model.predict(
        source=str(image_path),
        conf=MIN_PART_CONF,
        verbose=False
    )[0]

    damages = extract_boxes(damage_result, damage_model.names)
    parts = extract_boxes(carparts_result, carparts_model.names)

    yolo_labels = []

    for damage in damages:
        damage_box = damage["box"]
        damage_center = box_center(damage_box)

        best_match = None
        best_iou = 0.0

        for part in parts:
            part_name = part["class_name"]

            if part_name not in PART_TO_DAMAGED_CLASS:
                continue

            part_box = part["box"]
            iou = calculate_iou(damage_box, part_box)
            center_match = center_inside_box(damage_center, part_box)

            if center_match or iou >= MIN_IOU:
                if iou > best_iou:
                    best_iou = iou
                    best_match = part

        if best_match is None:
            continue

        damaged_class_name = PART_TO_DAMAGED_CLASS[best_match["class_name"]]
        class_id = CLASS_TO_ID[damaged_class_name]

        # Use the matched VEHICLE PART box as the label box.
        # This helps the model learn the exact damaged repair part,
        # not only the small damage patch.
        label_box = best_match["box"]

        x_center, y_center, width, height = yolo_box_from_xyxy(
            label_box,
            image_width,
            image_height
        )

        # Avoid invalid tiny boxes
        if width <= 0 or height <= 0:
            continue

        yolo_labels.append(
            f"{class_id} {x_center:.6f} {y_center:.6f} {width:.6f} {height:.6f}"
        )

    if not yolo_labels:
        skipped_no_label += 1
        continue

    output_image_name = f"auto_{idx:05d}_{image_path.name}"
    output_label_name = Path(output_image_name).with_suffix(".txt").name

    shutil.copy2(image_path, OUTPUT_DIR / "images" / split / output_image_name)

    with open(OUTPUT_DIR / "labels" / split / output_label_name, "w", encoding="utf-8") as f:
        f.write("\n".join(yolo_labels))

    created_images += 1
    created_labels += len(yolo_labels)

print("Auto-labelled images created:", created_images)
print("YOLO labels created:", created_labels)
print("Images skipped because no reliable label:", skipped_no_label)

Auto-labelled images created: 446
YOLO labels created: 769
Images skipped because no reliable label: 1553


In [15]:
yaml_path = OUTPUT_DIR / "data.yaml"

names_text = "\n".join([f"  {idx}: {name}" for idx, name in enumerate(DAMAGED_PART_CLASSES)])

yaml_content = f"""path: {OUTPUT_DIR.as_posix()}
train: images/train
val: images/val

names:
{names_text}
"""

with open(yaml_path, "w", encoding="utf-8") as f:
    f.write(yaml_content)

print("Created:", yaml_path)
print(yaml_content)

Created: C:\Users\User\Desktop\Vehicle_Damage_Detection\data\processed\damaged_parts_yolo\data.yaml
path: C:/Users/User/Desktop/Vehicle_Damage_Detection/data/processed/damaged_parts_yolo
train: images/train
val: images/val

names:
  0: front_bumper_damage
  1: rear_bumper_damage
  2: hood_damage
  3: front_light_damage
  4: rear_light_damage
  5: front_door_damage
  6: rear_door_damage
  7: side_mirror_damage
  8: windshield_damage
  9: wheel_tire_damage



In [16]:
for split in ["train", "val"]:
    image_count = len(list((OUTPUT_DIR / "images" / split).glob("*.*")))
    label_count = len(list((OUTPUT_DIR / "labels" / split).glob("*.txt")))

    print(f"\n=== {split.upper()} ===")
    print("Images:", image_count)
    print("Labels:", label_count)


=== TRAIN ===
Images: 361
Labels: 361

=== VAL ===
Images: 85
Labels: 85


In [17]:
from collections import Counter
from pathlib import Path

label_files = list((OUTPUT_DIR / "labels" / "train").glob("*.txt")) + list((OUTPUT_DIR / "labels" / "val").glob("*.txt"))

class_counter = Counter()

for label_file in label_files:
    with open(label_file, "r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                class_id = int(line.split()[0])
                class_counter[class_id] += 1

print("Class distribution:")

for class_id, count in sorted(class_counter.items()):
    print(class_id, DAMAGED_PART_CLASSES[class_id], ":", count)

Class distribution:
0 front_bumper_damage : 196
1 rear_bumper_damage : 299
2 hood_damage : 73
3 front_light_damage : 18
4 rear_light_damage : 20
5 front_door_damage : 13
6 rear_door_damage : 24
7 side_mirror_damage : 4
8 windshield_damage : 43
9 wheel_tire_damage : 79


In [18]:
from PIL import Image, ImageEnhance, ImageOps, ImageFilter
from pathlib import Path
from collections import Counter
import shutil

TRAIN_IMG_DIR = OUTPUT_DIR / "images" / "train"
TRAIN_LABEL_DIR = OUTPUT_DIR / "labels" / "train"

# Weak classes we want to improve
TARGET_CLASS_MINIMUMS = {
    3: 50,  # front_light_damage
    4: 50,  # rear_light_damage
    5: 50,  # front_door_damage
    6: 50,  # rear_door_damage
    7: 40,  # side_mirror_damage
}

def read_label_file(label_path):
    rows = []

    with open(label_path, "r", encoding="utf-8") as file:
        for line in file:
            parts = line.strip().split()

            if len(parts) != 5:
                continue

            class_id = int(parts[0])
            x_center = float(parts[1])
            y_center = float(parts[2])
            width = float(parts[3])
            height = float(parts[4])

            rows.append([class_id, x_center, y_center, width, height])

    return rows


def write_label_file(label_path, rows):
    lines = []

    for row in rows:
        class_id, x_center, y_center, width, height = row
        lines.append(
            f"{class_id} {x_center:.6f} {y_center:.6f} {width:.6f} {height:.6f}"
        )

    with open(label_path, "w", encoding="utf-8") as file:
        file.write("\n".join(lines))


def flip_labels_horizontally(rows):
    """
    For horizontal flip, x_center becomes 1 - x_center.
    """
    flipped_rows = []

    for row in rows:
        class_id, x_center, y_center, width, height = row
        flipped_rows.append([class_id, 1.0 - x_center, y_center, width, height])

    return flipped_rows


def count_train_classes():
    counter = Counter()

    for label_path in TRAIN_LABEL_DIR.glob("*.txt"):
        rows = read_label_file(label_path)

        for row in rows:
            counter[row[0]] += 1

    return counter


def find_image_for_label(label_path):
    image_stem = label_path.stem

    for ext in [".jpg", ".jpeg", ".png"]:
        image_path = TRAIN_IMG_DIR / f"{image_stem}{ext}"

        if image_path.exists():
            return image_path

    return None


current_counts = count_train_classes()

print("Before augmentation:")
for class_id, count in sorted(current_counts.items()):
    print(class_id, DAMAGED_PART_CLASSES[class_id], ":", count)


label_files = list(TRAIN_LABEL_DIR.glob("*.txt"))

augmented_images = 0
augmented_labels = 0

for label_path in label_files:
    rows = read_label_file(label_path)
    class_ids_in_image = {row[0] for row in rows}

    # Only augment images that contain weak classes
    useful_for_weak_class = False

    for target_class_id, target_minimum in TARGET_CLASS_MINIMUMS.items():
        if target_class_id in class_ids_in_image and current_counts[target_class_id] < target_minimum:
            useful_for_weak_class = True
            break

    if not useful_for_weak_class:
        continue

    image_path = find_image_for_label(label_path)

    if image_path is None:
        continue

    try:
        img = Image.open(image_path).convert("RGB")
    except Exception:
        continue

    augmentations = [
        ("bright", ImageEnhance.Brightness(img).enhance(1.25), rows),
        ("dark", ImageEnhance.Brightness(img).enhance(0.75), rows),
        ("contrast", ImageEnhance.Contrast(img).enhance(1.30), rows),
        ("sharp", ImageEnhance.Sharpness(img).enhance(1.50), rows),
        ("blur", img.filter(ImageFilter.GaussianBlur(radius=0.6)), rows),
        ("flip", ImageOps.mirror(img), flip_labels_horizontally(rows)),
    ]

    for aug_name, aug_img, aug_rows in augmentations:
        # Stop if all target classes reached enough samples
        all_done = True

        for target_class_id, target_minimum in TARGET_CLASS_MINIMUMS.items():
            if current_counts[target_class_id] < target_minimum:
                all_done = False
                break

        if all_done:
            break

        output_image_name = f"aug_{aug_name}_{image_path.stem}{image_path.suffix}"
        output_label_name = f"aug_{aug_name}_{label_path.stem}.txt"

        output_image_path = TRAIN_IMG_DIR / output_image_name
        output_label_path = TRAIN_LABEL_DIR / output_label_name

        # avoid duplicates
        if output_image_path.exists() or output_label_path.exists():
            continue

        aug_img.save(output_image_path)
        write_label_file(output_label_path, aug_rows)

        augmented_images += 1
        augmented_labels += len(aug_rows)

        for row in aug_rows:
            current_counts[row[0]] += 1

print("\nAugmented images created:", augmented_images)
print("Augmented labels created:", augmented_labels)

print("\nAfter augmentation:")
for class_id, count in sorted(current_counts.items()):
    print(class_id, DAMAGED_PART_CLASSES[class_id], ":", count)

Before augmentation:
0 front_bumper_damage : 171
1 rear_bumper_damage : 241
2 hood_damage : 61
3 front_light_damage : 14
4 rear_light_damage : 17
5 front_door_damage : 8
6 rear_door_damage : 22
7 side_mirror_damage : 3
8 windshield_damage : 40
9 wheel_tire_damage : 74

Augmented images created: 144
Augmented labels created: 294

After augmentation:
0 front_bumper_damage : 189
1 rear_bumper_damage : 283
2 hood_damage : 61
3 front_light_damage : 50
4 rear_light_damage : 53
5 front_door_damage : 50
6 rear_door_damage : 64
7 side_mirror_damage : 21
8 windshield_damage : 58
9 wheel_tire_damage : 116


In [19]:
from pathlib import Path
from PIL import Image
import random
import shutil
import os
import sys
from ultralytics import YOLO

PROJECT_ROOT = Path(r"C:\Users\User\Desktop\Vehicle_Damage_Detection")
os.chdir(PROJECT_ROOT)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

DAMAGE_MODEL_PATH = PROJECT_ROOT / "models" / "detection" / "cardd_coco_best.pt"
CARPARTS_MODEL_PATH = PROJECT_ROOT / "models" / "segmentation" / "carparts_best.pt"

OUTPUT_DIR = PROJECT_ROOT / "data" / "processed" / "damaged_parts_yolo"

SOURCE_IMAGE_DIRS = [
    PROJECT_ROOT / "data" / "processed" / "cardd_coco_yolo" / "images" / "train",
    PROJECT_ROOT / "data" / "processed" / "cardd_coco_yolo" / "images" / "val",
    PROJECT_ROOT / "data" / "processed" / "cardd_sod_yolo" / "images" / "train",
    PROJECT_ROOT / "data" / "processed" / "cardd_sod_yolo" / "images" / "val",
    PROJECT_ROOT / "data" / "processed" / "archive4_yolo" / "images" / "train",
    PROJECT_ROOT / "data" / "processed" / "archive4_yolo" / "images" / "val",
]

DAMAGED_PART_CLASSES = [
    "front_bumper_damage",
    "rear_bumper_damage",
    "hood_damage",
    "front_light_damage",
    "rear_light_damage",
    "front_door_damage",
    "rear_door_damage",
    "side_mirror_damage",
    "windshield_damage",
    "wheel_tire_damage",
]

CLASS_TO_ID = {name: idx for idx, name in enumerate(DAMAGED_PART_CLASSES)}

print("Damage model exists:", DAMAGE_MODEL_PATH.exists())
print("Carparts model exists:", CARPARTS_MODEL_PATH.exists())
print("Output dir exists:", OUTPUT_DIR.exists())
print("CLASS_TO_ID:", CLASS_TO_ID)

damage_model = YOLO(str(DAMAGE_MODEL_PATH))
carparts_model = YOLO(str(CARPARTS_MODEL_PATH))

print("Models loaded")

Damage model exists: True
Carparts model exists: True
Output dir exists: True
CLASS_TO_ID: {'front_bumper_damage': 0, 'rear_bumper_damage': 1, 'hood_damage': 2, 'front_light_damage': 3, 'rear_light_damage': 4, 'front_door_damage': 5, 'rear_door_damage': 6, 'side_mirror_damage': 7, 'windshield_damage': 8, 'wheel_tire_damage': 9}
Models loaded


In [20]:
def calculate_iou(box_a, box_b):
    x1_a, y1_a, x2_a, y2_a = box_a
    x1_b, y1_b, x2_b, y2_b = box_b

    inter_x1 = max(x1_a, x1_b)
    inter_y1 = max(y1_a, y1_b)
    inter_x2 = min(x2_a, x2_b)
    inter_y2 = min(y2_a, y2_b)

    inter_w = max(0, inter_x2 - inter_x1)
    inter_h = max(0, inter_y2 - inter_y1)
    intersection = inter_w * inter_h

    area_a = max(0, x2_a - x1_a) * max(0, y2_a - y1_a)
    area_b = max(0, x2_b - x1_b) * max(0, y2_b - y1_b)

    union = area_a + area_b - intersection

    if union == 0:
        return 0.0

    return intersection / union


def box_center(box):
    x1, y1, x2, y2 = box
    return [(x1 + x2) / 2, (y1 + y2) / 2]


def center_inside_box(center, box):
    cx, cy = center
    x1, y1, x2, y2 = box
    return x1 <= cx <= x2 and y1 <= cy <= y2


def yolo_box_from_xyxy(box, image_width, image_height):
    x1, y1, x2, y2 = box

    x_center = ((x1 + x2) / 2) / image_width
    y_center = ((y1 + y2) / 2) / image_height
    width = (x2 - x1) / image_width
    height = (y2 - y1) / image_height

    return x_center, y_center, width, height


def extract_boxes(result, names):
    items = []

    if result.boxes is None:
        return items

    for box in result.boxes:
        class_id = int(box.cls[0])
        confidence = float(box.conf[0])
        xyxy = box.xyxy[0].tolist()

        items.append({
            "class_name": names[class_id],
            "confidence": confidence,
            "box": xyxy
        })

    return items

print("Helper functions loaded")

Helper functions loaded


In [21]:
from pathlib import Path
from PIL import Image
import random
import shutil

MIRROR_CLASS_ID = CLASS_TO_ID["side_mirror_damage"]

MIRROR_OUTPUT_IMG_TRAIN = OUTPUT_DIR / "images" / "train"
MIRROR_OUTPUT_LABEL_TRAIN = OUTPUT_DIR / "labels" / "train"

# Use lower thresholds only for mirror because mirrors are small
MIRROR_DAMAGE_CONF = 0.20
MIRROR_PART_CONF = 0.10
MIRROR_MIN_IOU = 0.02

mirror_created = 0
mirror_labels_created = 0

# Use more images if available
mirror_image_paths = []

for folder in SOURCE_IMAGE_DIRS:
    if folder.exists():
        mirror_image_paths.extend(list(folder.glob("*.jpg")))
        mirror_image_paths.extend(list(folder.glob("*.jpeg")))
        mirror_image_paths.extend(list(folder.glob("*.png")))

random.shuffle(mirror_image_paths)
mirror_image_paths = mirror_image_paths[:3000]

for idx, image_path in enumerate(mirror_image_paths):
    try:
        image = Image.open(image_path).convert("RGB")
        image_width, image_height = image.size
    except Exception:
        continue

    damage_result = damage_model.predict(
        source=str(image_path),
        conf=MIRROR_DAMAGE_CONF,
        verbose=False
    )[0]

    carparts_result = carparts_model.predict(
        source=str(image_path),
        conf=MIRROR_PART_CONF,
        verbose=False
    )[0]

    damages = extract_boxes(damage_result, damage_model.names)
    parts = extract_boxes(carparts_result, carparts_model.names)

    mirror_labels = []

    for damage in damages:
        damage_box = damage["box"]
        damage_center = box_center(damage_box)

        for part in parts:
            part_name = part["class_name"]

            if part_name not in ["left_mirror", "right_mirror"]:
                continue

            part_box = part["box"]
            iou = calculate_iou(damage_box, part_box)
            center_match = center_inside_box(damage_center, part_box)

            if center_match or iou >= MIRROR_MIN_IOU:
                # Use mirror part box as label box
                x_center, y_center, width, height = yolo_box_from_xyxy(
                    part_box,
                    image_width,
                    image_height
                )

                if width <= 0 or height <= 0:
                    continue

                mirror_labels.append(
                    f"{MIRROR_CLASS_ID} {x_center:.6f} {y_center:.6f} {width:.6f} {height:.6f}"
                )

    if not mirror_labels:
        continue

    output_image_name = f"mirror_auto_{idx:05d}_{image_path.name}"
    output_label_name = Path(output_image_name).with_suffix(".txt").name

    output_image_path = MIRROR_OUTPUT_IMG_TRAIN / output_image_name
    output_label_path = MIRROR_OUTPUT_LABEL_TRAIN / output_label_name

    if output_image_path.exists() or output_label_path.exists():
        continue

    shutil.copy2(image_path, output_image_path)

    with open(output_label_path, "w", encoding="utf-8") as f:
        f.write("\n".join(mirror_labels))

    mirror_created += 1
    mirror_labels_created += len(mirror_labels)

print("Mirror-focused images created:", mirror_created)
print("Mirror-focused labels created:", mirror_labels_created)

Mirror-focused images created: 29
Mirror-focused labels created: 41


In [22]:
from collections import Counter

def count_classes_for_split(split):
    label_dir = OUTPUT_DIR / "labels" / split
    counter = Counter()

    for label_file in label_dir.glob("*.txt"):
        with open(label_file, "r", encoding="utf-8") as f:
            for line in f:
                if line.strip():
                    class_id = int(line.split()[0])
                    counter[class_id] += 1

    return counter


train_counter = count_classes_for_split("train")
val_counter = count_classes_for_split("val")

total_counter = Counter()
total_counter.update(train_counter)
total_counter.update(val_counter)

print("=== TRAIN CLASS DISTRIBUTION ===")
for class_id, class_name in enumerate(DAMAGED_PART_CLASSES):
    print(class_id, class_name, ":", train_counter[class_id])

print("\n=== VAL CLASS DISTRIBUTION ===")
for class_id, class_name in enumerate(DAMAGED_PART_CLASSES):
    print(class_id, class_name, ":", val_counter[class_id])

print("\n=== TOTAL CLASS DISTRIBUTION ===")
for class_id, class_name in enumerate(DAMAGED_PART_CLASSES):
    print(class_id, class_name, ":", total_counter[class_id])

=== TRAIN CLASS DISTRIBUTION ===
0 front_bumper_damage : 189
1 rear_bumper_damage : 283
2 hood_damage : 61
3 front_light_damage : 50
4 rear_light_damage : 53
5 front_door_damage : 50
6 rear_door_damage : 64
7 side_mirror_damage : 62
8 windshield_damage : 58
9 wheel_tire_damage : 116

=== VAL CLASS DISTRIBUTION ===
0 front_bumper_damage : 25
1 rear_bumper_damage : 58
2 hood_damage : 12
3 front_light_damage : 4
4 rear_light_damage : 3
5 front_door_damage : 5
6 rear_door_damage : 2
7 side_mirror_damage : 1
8 windshield_damage : 3
9 wheel_tire_damage : 5

=== TOTAL CLASS DISTRIBUTION ===
0 front_bumper_damage : 214
1 rear_bumper_damage : 341
2 hood_damage : 73
3 front_light_damage : 54
4 rear_light_damage : 56
5 front_door_damage : 55
6 rear_door_damage : 66
7 side_mirror_damage : 63
8 windshield_damage : 61
9 wheel_tire_damage : 121


In [23]:
for split in ["train", "val"]:
    image_count = len(list((OUTPUT_DIR / "images" / split).glob("*.*")))
    label_count = len(list((OUTPUT_DIR / "labels" / split).glob("*.txt")))

    print(f"\n=== {split.upper()} ===")
    print("Images:", image_count)
    print("Labels:", label_count)


=== TRAIN ===
Images: 534
Labels: 534

=== VAL ===
Images: 85
Labels: 85
